<a href="https://colab.research.google.com/github/ChungKiet/DA_ebay_sale_car/blob/main/ebay_car_sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Introduction

This dataset holds 50,000 used-car listings scraped from eBay Kleinanzeigen, the classifieds section of the German eBay site. Each row describes one listing: when it was crawled, the asking price, vehicle type, registration year, mileage, fuel type, brand, and whether the car has unrepaired damage, among other fields. The raw data was left intentionally messy for this project (inconsistent column names, numeric values stored as text, and a handful of clearly wrong entries), so the first several sections below focus on cleaning it before moving on to the price analysis.

In [44]:
import numpy as np
import pandas as pd

In [5]:
car_sales = pd.read_csv('autos.csv', encoding='Latin-1')
car_sales.info()
car_sales.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   dateCrawled          50000 non-null  object
 1   name                 50000 non-null  object
 2   seller               50000 non-null  object
 3   offerType            50000 non-null  object
 4   price                50000 non-null  object
 5   abtest               50000 non-null  object
 6   vehicleType          44905 non-null  object
 7   yearOfRegistration   50000 non-null  int64 
 8   gearbox              47320 non-null  object
 9   powerPS              50000 non-null  int64 
 10  model                47242 non-null  object
 11  odometer             50000 non-null  object
 12  monthOfRegistration  50000 non-null  int64 
 13  fuelType             45518 non-null  object
 14  brand                50000 non-null  object
 15  notRepairedDamage    40171 non-null  object
 16  date

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,odometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
0,2016-03-26 17:47:46,Peugeot_807_160_NAVTECH_ON_BOARD,privat,Angebot,"$5,000",control,bus,2004,manuell,158,andere,"150,000km",3,lpg,peugeot,nein,2016-03-26 00:00:00,0,79588,2016-04-06 06:45:54
1,2016-04-04 13:38:56,BMW_740i_4_4_Liter_HAMANN_UMBAU_Mega_Optik,privat,Angebot,"$8,500",control,limousine,1997,automatik,286,7er,"150,000km",6,benzin,bmw,nein,2016-04-04 00:00:00,0,71034,2016-04-06 14:45:08
2,2016-03-26 18:57:24,Volkswagen_Golf_1.6_United,privat,Angebot,"$8,990",test,limousine,2009,manuell,102,golf,"70,000km",7,benzin,volkswagen,nein,2016-03-26 00:00:00,0,35394,2016-04-06 20:15:37
3,2016-03-12 16:58:10,Smart_smart_fortwo_coupe_softouch/F1/Klima/Pan...,privat,Angebot,"$4,350",control,kleinwagen,2007,automatik,71,fortwo,"70,000km",6,benzin,smart,nein,2016-03-12 00:00:00,0,33729,2016-03-15 03:16:28
4,2016-04-01 14:38:50,Ford_Focus_1_6_Benzin_TÜV_neu_ist_sehr_gepfleg...,privat,Angebot,"$1,350",test,kombi,2003,manuell,0,focus,"150,000km",7,benzin,ford,nein,2016-04-01 00:00:00,0,39218,2016-04-01 14:38:50


The output above already hints at the cleanup work ahead: several columns (vehicleType, gearbox, model, fuelType, notRepairedDamage) have noticeably fewer than 50,000 non-null values, so missing data will need to be handled, and the column names themselves are still in camelCase rather than the snake_case we will switch to next.

## 2. Cleaning columns name

### Overview data
- 50000 rows
- 20 columns
- Columnn name should camel case now, should rename to snake case

In [7]:
car_sales.columns

Index(['dateCrawled', 'name', 'seller', 'offerType', 'price', 'abtest',
       'vehicleType', 'yearOfRegistration', 'gearbox', 'powerPS', 'model',
       'odometer', 'monthOfRegistration', 'fuelType', 'brand',
       'notRepairedDamage', 'dateCreated', 'nrOfPictures', 'postalCode',
       'lastSeen'],
      dtype='object')

In [27]:
car_sales.rename({
              'dateCrawled':'date_crawled',
              'offerType':'offer_type',
              'vehicleType':'vehicle_type',
              'yearOfRegistration':'registration_year',
              'powerPS':'CV',
              'monthOfRegistration':'registration_month',
              'fuelType':'fuel_type',
              'notRepairedDamage':'unrepared_damage',
              'dateCreated':'ad_created',
              'nrOfPictures' : 'nr_pictures',
              'price':'price_in_dollars',
              'postalCode':'postal_code',
              'odometer':'odometer_km',
              'lastSeen':'last_seen',}, axis = 1, inplace = True)

In [28]:
car_sales.columns

Index(['date_crawled', 'name', 'seller', 'offer_type', 'price_in_dollars',
       'abtest', 'vehicle_type', 'registration_year', 'gearbox', 'CV', 'model',
       'odometer_km', 'registration_month', 'fuel_type', 'brand',
       'unrepared_damage', 'ad_created', 'nr_pictures', 'postal_code',
       'last_seen'],
      dtype='object')

All 20 columns are now snake_case and match the naming style used for the rest of this analysis (for example price_in_dollars, odometer_km and unrepared_damage), which makes them much easier to reference in code going forward.

## 3. Initial Cleaning

In [10]:
car_sales.describe(include = 'all')

,date_crawled,name,seller,offer_type,price_in_dollars,abtest,vehicle_type,registration_year,gearbox,CV,model,odometer,registration_month,fuel_type,brand,unrepared_damage,ad_created,nr_pictures,postal_code,last_seen
count,50000,50000,50000,50000,50000,50000,44905,50000.000000,47320,50000.000000,47242,50000,50000.000000,45518,50000,40171,50000,50000.0,50000.000000,50000
unique,48213,38754,2,2,2357,2,8,NaN,2,NaN,245,13,NaN,7,40,2,76,NaN,NaN,39481
top,2016-03-19 17:36:18,Ford_Fiesta,privat,Angebot,$0,test,limousine,NaN,manuell,NaN,golf,"150,000km",NaN,benzin,volkswagen,nein,2016-04-03 00:00:00,NaN,NaN,2016-04-07 06:17:27
freq,3,78,49999,49999,1421,25756,12859,NaN,36993,NaN,4024,32424,NaN,30107,10687,35232,1946,NaN,NaN,8
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2005.073280,NaN,116.355920,NaN,NaN,5.723360,NaN,NaN,NaN,NaN,0.0,50813.627300,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.712813,NaN,209.216627,NaN,NaN,3.711984,NaN,NaN,NaN,NaN,0.0,25779.747957,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1000.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,0.0,1067.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1999.000000,NaN,70.000000,NaN,NaN,3.000000,NaN,NaN,NaN,NaN,0.0,30451.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003.000000,NaN,105.000000,NaN,NaN,6.000000,NaN,NaN,NaN,NaN,0.0,49577.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.000000,NaN,150.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,0.0,71540.000000,NaN


In [11]:
car_sales['abtest'].value_counts()

,count
abtest,
test,25756
control,24244


In [12]:
car_sales['offer_type'].value_counts()

,count
offer_type,
Angebot,49999
Gesuch,1


In [15]:
car_sales['seller'].value_counts()

,count
nr_pictures,
0,50000


In [16]:
car_sales['nr_pictures'].value_counts()

,count
nr_pictures,
0,50000


## Dataset observations:

- The raw dataset holds 50,000 listings across 20 columns, and none of the column names follow snake_case, so they were renamed for consistency before doing anything else.
- `seller`, `offer_type` and `nr_pictures` are almost single-valued (49,999/50,000 records share one value, and every listing has 0 pictures), so they carry almost no predictive information and are candidates to drop later.
- `abtest` is split roughly 51/49 between `test` and `control`, which looks like a genuine experiment flag rather than noise, so it is worth keeping for now.
- `price_in_dollars` and `odometer_km` are numeric in nature but stored as text (with `$`, commas and the `km` suffix), so they need to be cleaned and cast to integers before any numeric analysis can happen.

In [19]:
car_sales['price_in_dollars'][:5]

,price_in_dollars
0,"$5,000"
1,"$8,500"
2,"$8,990"
3,"$4,350"
4,"$1,350"


In [26]:
car_sales['price_in_dollars'] = car_sales['price_in_dollars'].str.replace('$', '')
car_sales['price_in_dollars'] = car_sales['price_in_dollars'].str.replace(',', '')
car_sales['price_in_dollars'] = car_sales['price_in_dollars'].astype(int)

In [25]:
car_sales['price_in_dollars'][:5]

,price_in_dollars
0,5000
1,8500
2,8990
3,4350
4,1350


In [30]:
car_sales['odometer_km'][:5]

,odometer_km
0,"150,000km"
1,"150,000km"
2,"70,000km"
3,"70,000km"
4,"150,000km"


In [31]:
car_sales['odometer_km'] = car_sales['odometer_km'].str.replace('km', '')
car_sales['odometer_km'] = car_sales['odometer_km'].str.replace(',', '')
car_sales['odometer_km'] = car_sales['odometer_km'].astype(int)

In [32]:
car_sales['odometer_km'][:5]

,odometer_km
0,150000
1,150000
2,70000
3,70000
4,150000


## 4. Exploring the odometer_km and price columns

The summary below shows a mean price of roughly 9,840 dollars but a maximum of nearly 100 million, which is impossible for a used car, so this section works through both ends of the price distribution (and double-checks the mileage column) to decide where to draw sensible cut-off points.

In [36]:
car_sales['price_in_dollars'].describe().round()

,price_in_dollars
count,50000.0
mean,9840.0
std,481104.0
min,0.0
25%,1100.0
50%,2950.0
75%,7200.0
max,99999999.0


In [41]:
car_sales['price_in_dollars'].value_counts().sort_index()

,count
price_in_dollars,
0,1421
1,156
2,3
3,1
5,2
...,...
10000000,1
11111111,2
12345678,3


In [39]:
top_List = car_sales[(car_sales['price_in_dollars'] >= 265000)].sort_index(ascending = False)
top_List

,date_crawled,name,seller,offer_type,price_in_dollars,abtest,vehicle_type,registration_year,gearbox,CV,model,odometer_km,registration_month,fuel_type,brand,unrepared_damage,ad_created,nr_pictures,postal_code,last_seen
47634,2016-04-04 21:25:21,Ferrari_FXX,privat,Angebot,3890000,test,coupe,2006,NaN,799,NaN,5000,7,NaN,sonstige_autos,nein,2016-04-04 00:00:00,0,60313,2016-04-05 12:07:37
47598,2016-03-31 18:56:54,Opel_Vectra_B_1_6i_16V_Facelift_Tuning_Showcar...,privat,Angebot,12345678,control,limousine,2001,manuell,101,vectra,150000,3,benzin,opel,nein,2016-03-31 00:00:00,0,4356,2016-03-31 18:56:54
43049,2016-03-21 19:53:52,2_VW_Busse_T3,privat,Angebot,999999,test,bus,1981,manuell,70,transporter,150000,1,benzin,volkswagen,NaN,2016-03-21 00:00:00,0,99880,2016-03-28 17:18:28
42221,2016-03-08 20:39:05,Leasinguebernahme,privat,Angebot,27322222,control,limousine,2014,manuell,163,c4,40000,2,diesel,citroen,NaN,2016-03-08 00:00:00,0,76532,2016-03-08 20:39:05
39705,2016-03-22 14:58:27,Tausch_gegen_gleichwertiges,privat,Angebot,99999999,control,limousine,1999,automatik,224,s_klasse,150000,9,benzin,mercedes_benz,NaN,2016-03-22 00:00:00,0,73525,2016-04-06 05:15:30
39377,2016-03-08 23:53:51,Tausche_volvo_v40_gegen_van,privat,Angebot,12345678,control,NaN,2018,manuell,95,v40,150000,6,NaN,volvo,nein,2016-03-08 00:00:00,0,14542,2016-04-06 23:17:31
37585,2016-03-29 11:38:54,Volkswagen_Jetta_GT,privat,Angebot,999990,test,limousine,1985,manuell,111,jetta,150000,12,benzin,volkswagen,ja,2016-03-29 00:00:00,0,50997,2016-03-29 11:38:54
36818,2016-03-27 18:37:37,Porsche_991,privat,Angebot,350000,control,coupe,2016,manuell,500,911,5000,3,benzin,porsche,nein,2016-03-27 00:00:00,0,70499,2016-03-27 18:37:37
35923,2016-04-03 07:56:23,Porsche_911_Targa_Exclusive_Edition__1_von_15_...,privat,Angebot,295000,test,cabrio,2015,automatik,400,911,5000,6,benzin,porsche,nein,2016-04-03 00:00:00,0,74078,2016-04-03 08:56:20
34723,2016-03-23 16:37:29,Porsche_Porsche_911/930_Turbo_3.0__deutsche_Au...,privat,Angebot,299000,test,coupe,1977,manuell,260,911,100000,7,benzin,porsche,nein,2016-03-23 00:00:00,0,61462,2016-04-06 16:44:50


Scrolling through every column above is hard to compare, so next we narrow the view down to just price, brand and model. A few values immediately look like data-entry mistakes rather than real prices, such as 12345678, 11111111 and 99999999, which are suspiciously round or repeated-digit numbers rather than anything a seller would realistically type.

In [45]:
top_List[['price_in_dollars','brand', 'model']].sort_index(ascending = False)

,price_in_dollars,brand,model
47634,3890000,sonstige_autos,NaN
47598,12345678,opel,vectra
43049,999999,volkswagen,transporter
42221,27322222,citroen,c4
39705,99999999,mercedes_benz,s_klasse
39377,12345678,volvo,v40
37585,999990,volkswagen,jetta
36818,350000,porsche,911
35923,295000,porsche,911
34723,299000,porsche,911


In [46]:
top_List[top_List['brand'] == 'sonstige_autos']

,date_crawled,name,seller,offer_type,price_in_dollars,abtest,vehicle_type,registration_year,gearbox,CV,model,odometer_km,registration_month,fuel_type,brand,unrepared_damage,ad_created,nr_pictures,postal_code,last_seen
47634,2016-04-04 21:25:21,Ferrari_FXX,privat,Angebot,3890000,test,coupe,2006,NaN,799,NaN,5000,7,NaN,sonstige_autos,nein,2016-04-04 00:00:00,0,60313,2016-04-05 12:07:37
14715,2016-03-30 08:37:24,Rolls_Royce_Phantom_Drophead_Coupe,privat,Angebot,345000,control,cabrio,2012,automatik,460,NaN,20000,8,benzin,sonstige_autos,nein,2016-03-30 00:00:00,0,73525,2016-04-07 00:16:26
11137,2016-03-29 23:52:57,suche_maserati_3200_gt_Zustand_unwichtig_laufe...,privat,Angebot,10000000,control,coupe,1960,manuell,368,NaN,100000,1,benzin,sonstige_autos,nein,2016-03-29 00:00:00,0,73033,2016-04-06 21:18:11
7814,2016-04-04 11:53:31,Ferrari_F40,privat,Angebot,1300000,control,coupe,1992,NaN,0,NaN,50000,12,NaN,sonstige_autos,nein,2016-04-04 00:00:00,0,60598,2016-04-05 11:34:11


Among the sonstige_autos (other cars) rows above is a listing titled roughly "looking for a maserati 3200 gt, condition unimportant" priced at 10,000,000 dollars, which is obviously not a real asking price. The registration year on file (1960) does not match the actual 3200 GT, a model built between 1998 and 2001, but it is consistent with the earlier 3500 GT produced from 1957 to 1964. Real listings for that classic model run roughly between 150,000 and 860,000 dollars, so we replace the bogus price with 507,230, the midpoint of that range, rather than dropping the row entirely.

In [48]:
car_sales.iloc[11137,3] = 507230

In [50]:
car_sales.iloc[11137,:]

,11137
date_crawled,2016-03-29 23:52:57
name,suche_maserati_3200_gt_Zustand_unwichtig_laufe...
seller,privat
offer_type,507230
price_in_dollars,10000000
abtest,control
vehicle_type,coupe
registration_year,1960
gearbox,manuell
CV,368


Looking back at the narrowed table, the sonstige_autos and porsche listings in this high-price range are the ones with plausible names and models (Ferraris, a Rolls-Royce, several Porsche 911s), so those brands are kept as genuine. Everything else in this top slice, mostly repeated-digit prices like 999999 or 12345678 attached to ordinary cars such as an Opel Vectra or a VW van, looks like a typo or placeholder value rather than a real price, so those rows are flagged for removal.

In [52]:
remove_bad_price_bool = ((top_List['brand'] != 'sonstige_autos') & (top_List['brand'] != 'porsche'))
remove_bad_price_bool.head(5)

,brand
47634,False
47598,True
43049,True
42221,True
39705,True


In [56]:
bad_cars = top_List[remove_bad_price_bool]
print('Length of bad cars: ', len(bad_cars))
bad_cars

Length of bad cars:  11


,date_crawled,name,seller,offer_type,price_in_dollars,abtest,vehicle_type,registration_year,gearbox,CV,model,odometer_km,registration_month,fuel_type,brand,unrepared_damage,ad_created,nr_pictures,postal_code,last_seen
47598,2016-03-31 18:56:54,Opel_Vectra_B_1_6i_16V_Facelift_Tuning_Showcar...,privat,Angebot,12345678,control,limousine,2001,manuell,101,vectra,150000,3,benzin,opel,nein,2016-03-31 00:00:00,0,4356,2016-03-31 18:56:54
43049,2016-03-21 19:53:52,2_VW_Busse_T3,privat,Angebot,999999,test,bus,1981,manuell,70,transporter,150000,1,benzin,volkswagen,NaN,2016-03-21 00:00:00,0,99880,2016-03-28 17:18:28
42221,2016-03-08 20:39:05,Leasinguebernahme,privat,Angebot,27322222,control,limousine,2014,manuell,163,c4,40000,2,diesel,citroen,NaN,2016-03-08 00:00:00,0,76532,2016-03-08 20:39:05
39705,2016-03-22 14:58:27,Tausch_gegen_gleichwertiges,privat,Angebot,99999999,control,limousine,1999,automatik,224,s_klasse,150000,9,benzin,mercedes_benz,NaN,2016-03-22 00:00:00,0,73525,2016-04-06 05:15:30
39377,2016-03-08 23:53:51,Tausche_volvo_v40_gegen_van,privat,Angebot,12345678,control,NaN,2018,manuell,95,v40,150000,6,NaN,volvo,nein,2016-03-08 00:00:00,0,14542,2016-04-06 23:17:31
37585,2016-03-29 11:38:54,Volkswagen_Jetta_GT,privat,Angebot,999990,test,limousine,1985,manuell,111,jetta,150000,12,benzin,volkswagen,ja,2016-03-29 00:00:00,0,50997,2016-03-29 11:38:54
27371,2016-03-09 15:45:47,Fiat_Punto,privat,Angebot,12345678,control,NaN,2017,NaN,95,punto,150000,0,NaN,fiat,NaN,2016-03-09 00:00:00,0,96110,2016-03-09 15:45:47
24384,2016-03-21 13:57:51,Schlachte_Golf_3_gt_tdi,privat,Angebot,11111111,test,NaN,1995,NaN,0,NaN,150000,0,NaN,volkswagen,NaN,2016-03-21 00:00:00,0,18519,2016-03-21 14:40:18
22947,2016-03-22 12:54:19,Bmw_530d_zum_ausschlachten,privat,Angebot,1234566,control,kombi,1999,automatik,190,NaN,150000,2,diesel,bmw,NaN,2016-03-22 00:00:00,0,17454,2016-04-02 03:17:32
2897,2016-03-12 21:50:57,Escort_MK_1_Hundeknochen_zum_umbauen_auf_RS_2000,privat,Angebot,11111111,test,limousine,1973,manuell,48,escort,50000,3,benzin,ford,nein,2016-03-12 00:00:00,0,94469,2016-03-12 22:45:27


In [57]:
for bad in bad_cars.index:
    car_sales.drop([bad], inplace = True)

In [58]:
car_sales.describe().round()

,price_in_dollars,registration_year,CV,odometer_km,registration_month,nr_pictures,postal_code
count,49989.0,49989.0,49989.0,49989.0,49989.0,49989.0,49989.0
mean,6025.0,2005.0,116.0,125732.0,6.0,0.0,50814.0
std,49134.0,106.0,209.0,40042.0,4.0,0.0,25777.0
min,0.0,1000.0,0.0,5000.0,0.0,0.0,1067.0
25%,1100.0,1999.0,70.0,125000.0,3.0,0.0,30451.0
50%,2950.0,2003.0,105.0,150000.0,6.0,0.0,49577.0
75%,7200.0,2008.0,150.0,150000.0,9.0,0.0,71522.0
max,10000000.0,9999.0,17700.0,150000.0,12.0,0.0,99998.0


Exploring price outliers on the bottom of the list:

In [59]:
car_sales['price_in_dollars'].describe().round()

,price_in_dollars
count,49989.0
mean,6025.0
std,49134.0
min,0.0
25%,1100.0
50%,2950.0
75%,7200.0
max,10000000.0


With those 11 clearly bogus rows dropped (49,989 rows remain), the mean price falls from 9,840 to about 6,025 dollars, which is far more believable, though the max is still 10,000,000 because of the Maserati listing we intentionally kept and re-priced. Next we look at the bottom end of the price range, since a price of 0 is just as unrealistic as a price of 100 million.

In [60]:
car_sales['price_in_dollars'].value_counts().sort_index(ascending = False)

,count
price_in_dollars,
10000000,1
3890000,1
1300000,1
350000,1
345000,1
...,...
5,2
3,1
2,3


In [61]:
car_sales[car_sales['price_in_dollars'].between(0,1100)].describe()

,price_in_dollars,registration_year,CV,odometer_km,registration_month,nr_pictures,postal_code
count,12538.000000,12538.000000,12538.000000,12538.000000,12538.000000,12538.0,12538.000000
mean,556.422396,2002.788563,74.677461,136162.466103,4.831552,0.0,47992.659595
std,339.147087,146.343951,174.629327,34988.383373,3.947714,0.0,25833.421398
min,0.000000,1111.000000,0.000000,5000.000000,0.000000,0.0,1069.000000
25%,300.000000,1996.000000,45.000000,150000.000000,1.000000,0.0,27376.250000
50%,599.000000,1999.000000,71.000000,150000.000000,4.000000,0.0,46240.000000
75%,850.000000,2001.000000,101.000000,150000.000000,8.000000,0.0,66482.000000
max,1100.000000,9999.000000,15016.000000,150000.000000,12.000000,0.0,99988.000000


1,421 listings are priced at exactly 0 and the 25th percentile only reaches 1,100, so a large slice of the data sits in this low range. Before deciding whether to drop it, it is worth checking how much money that slice actually represents compared with the rest of the dataset: if it is a tiny share of total sales, cutting it out will not meaningfully bias the analysis.

In [63]:
seventy_five = car_sales[car_sales['price_in_dollars'].between(850,1100)]
seventy_five.loc[:,'price_in_dollars'].sum()

np.int64(3227534)

In [65]:
fifty_percent = car_sales[car_sales['price_in_dollars'].between(599,850)]
fifty_percent.loc[:,'price_in_dollars'].sum()

np.int64(2462601)

In [68]:
quart = car_sales[car_sales['price_in_dollars'].between(300, 599)]
quart.loc[:,'price_in_dollars'].sum()

np.int64(1431649)

In [69]:
sum_total_dataset = car_sales[car_sales['price_in_dollars'].between(1100, 10000000)]
sum_total_dataset.loc[:,'price_in_dollars'].sum()

np.int64(294623333)

In [71]:
car_sales['odometer_km'].describe()

,odometer_km
count,49989.000000
mean,125732.061053
std,40042.171702
min,5000.000000
25%,125000.000000
50%,150000.000000
75%,150000.000000
max,150000.000000


Adding up the prices in each low band shows they contribute very little to total revenue: listings between 850 and 1,100 dollars total about 3.2 million, between 599 and 850 about 2.5 million, and between 300 and 599 about 1.4 million, compared to roughly 294.6 million dollars from everything priced above 1,100. Given how small these bands are in dollar terms, and that odometer_km itself looks reasonable (5,000 to 150,000 km, no obviously broken values), we settle on 850 dollars as the lower cutoff and keep the earlier upper bound of 10,000,000. This produces the car_cleaned copy that the rest of the notebook builds on, cutting the dataset from 50,000 to 40,786 rows.

In [72]:
car_cleaned = car_sales[car_sales['price_in_dollars'].between(850,10000000)].copy()

In [73]:
car_cleaned.describe().round()

,price_in_dollars,registration_year,CV,odometer_km,registration_month,nr_pictures,postal_code
count,40786.0,40786.0,40786.0,40786.0,40786.0,40786.0,40786.0
mean,7293.0,2005.0,127.0,123695.0,6.0,0.0,51561.0
std,54315.0,84.0,215.0,40347.0,4.0,0.0,25662.0
min,850.0,1000.0,0.0,5000.0,0.0,0.0,1067.0
25%,1950.0,2000.0,75.0,100000.0,3.0,0.0,31171.0
50%,3999.0,2005.0,116.0,150000.0,6.0,0.0,50739.0
75%,8500.0,2009.0,155.0,150000.0,9.0,0.0,72393.0
max,10000000.0,9999.0,17700.0,150000.0,12.0,0.0,99998.0


## 5. Exploring the data columns

These three columns record when each ad was first crawled, when it was created, and when it was last seen online, so looking at their distributions tells us over what calendar window this snapshot of the site was collected.

In [74]:
car_cleaned[['date_crawled','ad_created','last_seen']][:5]

,date_crawled,ad_created,last_seen
0,2016-03-26 17:47:46,2016-03-26 00:00:00,2016-04-06 06:45:54
1,2016-04-04 13:38:56,2016-04-04 00:00:00,2016-04-06 14:45:08
2,2016-03-26 18:57:24,2016-03-26 00:00:00,2016-04-06 20:15:37
3,2016-03-12 16:58:10,2016-03-12 00:00:00,2016-03-15 03:16:28
4,2016-04-01 14:38:50,2016-04-01 00:00:00,2016-04-01 14:38:50


In [75]:
car_cleaned['date_crawled'].str[:10].value_counts(normalize=True, dropna = False).sort_index()

,proportion
date_crawled,
2016-03-05,0.025573
2016-03-06,0.014147
2016-03-07,0.035600
2016-03-08,0.032683
2016-03-09,0.032462
2016-03-10,0.033075
2016-03-11,0.032658
2016-03-12,0.037415
2016-03-13,0.016035


In [76]:
car_cleaned['ad_created'].str[:10].value_counts(normalize=True, dropna=False).sort_index()

,proportion
ad_created,
2015-06-11,0.000025
2015-08-10,0.000025
2015-09-09,0.000025
2015-11-10,0.000025
2015-12-05,0.000025
...,...
2016-04-03,0.039425
2016-04-04,0.037194
2016-04-05,0.011965


In [77]:
car_cleaned['last_seen'].str[:10].value_counts(normalize=True, dropna=False).sort_index()

,proportion
last_seen,
2016-03-05,0.001128
2016-03-06,0.003800
2016-03-07,0.004781
2016-03-08,0.006522
2016-03-09,0.009121
2016-03-10,0.010126
2016-03-11,0.011622
2016-03-12,0.022753
2016-03-13,0.008581


Together these three date columns show the crawl ran mainly from early March to early April 2016, with date_crawled and ad_created spread fairly evenly across that window (a small number of ad_created values reach back to mid-2015, presumably older ads that were still active). last_seen is heavily concentrated in the final three days (04-05 through 04-07 account for about half of all listings), which makes sense since that is simply when the crawler stopped looking, catching most still-active ads on its last few passes rather than reflecting anything about the cars themselves.

## 6. Dealing with Incorrect Registration Year Data

A car's registration_year cannot logically be later than the year its ad was crawled (2016), and realistically should not be much earlier than the first mass-produced automobiles either, so this section checks both ends of that column for impossible values before deciding on sensible bounds.

In [78]:
car_cleaned['registration_year'].describe()

,registration_year
count,40786.000000
mean,2005.389864
std,84.381634
min,1000.000000
25%,2000.000000
50%,2005.000000
75%,2009.000000
max,9999.000000


In [80]:
car_cleaned['registration_year'].value_counts().sort_index(ascending=False).head(30)

,count
registration_year,
9999,2
9000,1
8888,1
6200,1
5911,1
5000,2
4500,1
4100,1
2800,1


Above 2019 the values stop making sense (9999, 9000, 8888, 6200, 5911, 5000, 4500, 4100 and 2800 all appear), and even 2019 itself is impossible since the data was crawled in 2016, so anything past 2016 should be treated as an error rather than a real registration date. Once we look past those outliers, the counts settle into a plausible yearly pattern from the late 1990s through 2016.

In [82]:
car_cleaned['registration_year'].value_counts().sort_index(ascending=False).tail(7)

,count
registration_year,
1937,4
1934,2
1931,1
1929,1
1927,1
1001,1
1000,1


At the low end, 1000 and 1001 are obvious placeholder or typo values, but 1927, 1929, 1931, 1934 and 1937 are all plausible: vintage cars do occasionally show up in classifieds. So we cut the dataset to registration years between 1927 and 2016 inclusive, keeping genuinely old cars while dropping the impossible entries at both extremes.

In [84]:
car_cleaned = car_cleaned[car_cleaned['registration_year'].between(1927, 2016)].copy()

In [85]:
car_cleaned['registration_year'].describe()

,registration_year
count,39224.000000
mean,2003.713619
std,7.022507
min,1927.000000
25%,2000.000000
50%,2004.000000
75%,2008.000000
max,2016.000000


In [86]:
car_cleaned['registration_year'].value_counts(normalize = True, dropna = False, bins = 20).sort_index(ascending = False).round(3)

,proportion
"(2011.55, 2016.0]",0.098
"(2007.1, 2011.55]",0.191
"(2002.65, 2007.1]",0.330
"(1998.2, 2002.65]",0.225
"(1993.75, 1998.2]",0.104
"(1989.3, 1993.75]",0.024
"(1984.85, 1989.3]",0.011
"(1980.4, 1984.85]",0.004
"(1975.95, 1980.4]",0.005
"(1971.5, 1975.95]",0.002


This binned view confirms that registrations are heavily weighted toward more recent cars: over 84% of listings fall between 1993 and 2011, and the 2002.65-2007.1 bucket alone accounts for a third of the cleaned dataset, while anything before the mid-1950s rounds to essentially zero percent. That is consistent with a used-car marketplace, where older vehicles naturally become rarer.

## 7. Exploring Price by Brand

With 39,224 rows spread across 40 different brands, most brands individually have too few listings to give a reliable average price. To keep the comparison meaningful, we focus on the 10 brands with the most listings and look at how their average price and average mileage compare.

In [87]:
top_brands = car_cleaned['brand'].value_counts()
top_brands

,count
brand,
volkswagen,8249
bmw,4761
mercedes_benz,4258
audi,3724
opel,3656
ford,2398
renault,1559
peugeot,1105
fiat,868


In [88]:
selected_brands = car_cleaned['brand'].value_counts().index[:10] # get top 10 brands
selected_brands

Index(['volkswagen', 'bmw', 'mercedes_benz', 'audi', 'opel', 'ford', 'renault',
       'peugeot', 'fiat', 'skoda'],
      dtype='object', name='brand')

In [89]:
brands_price = {}

for brand in selected_brands:
    sel_brand = car_cleaned[car_cleaned['brand'] == brand ]
    brands_price[brand] = sel_brand['price_in_dollars'].mean().round()

brands_price_sorted = sorted(brands_price.items(),key = lambda kv: kv[1],reverse=True) # sort desc
brands_price_sorted

[('audi', np.float64(10088.0)),
 ('mercedes_benz', np.float64(9094.0)),
 ('bmw', np.float64(8952.0)),
 ('skoda', np.float64(6739.0)),
 ('volkswagen', np.float64(6364.0)),
 ('ford', np.float64(4939.0)),
 ('opel', np.float64(3911.0)),
 ('peugeot', np.float64(3772.0)),
 ('fiat', np.float64(3710.0)),
 ('renault', np.float64(3297.0))]

In [90]:
brands_km = {}

for brand in sorted(selected_brands):
    sel_brand = car_cleaned[car_cleaned['brand'] == brand]
    brands_km[brand] = sel_brand['odometer_km'].mean().round()

brands_km_sorted = sorted(brands_km.items(), key = lambda kv: kv[1], reverse= True)
brands_km_sorted

[('bmw', np.float64(132200.0)),
 ('mercedes_benz', np.float64(130461.0)),
 ('audi', np.float64(128012.0)),
 ('volkswagen', np.float64(126561.0)),
 ('opel', np.float64(125495.0)),
 ('peugeot', np.float64(123362.0)),
 ('renault', np.float64(123307.0)),
 ('ford', np.float64(121253.0)),
 ('fiat', np.float64(110478.0)),
 ('skoda', np.float64(110430.0))]

In [91]:
listado = {}

template_string = "Mean price {money:.2f}$ and {km:.2f} mean Kilometers"

for vehiculo in brands_price:
    mean_price = brands_price[vehiculo]
    mean_km = brands_km[vehiculo]
    output = template_string.format(money = mean_price, km = mean_km )
    listado[vehiculo] = output


listado

{'volkswagen': 'Mean price 6364.00$ and 126561.00 mean Kilometers',
 'bmw': 'Mean price 8952.00$ and 132200.00 mean Kilometers',
 'mercedes_benz': 'Mean price 9094.00$ and 130461.00 mean Kilometers',
 'audi': 'Mean price 10088.00$ and 128012.00 mean Kilometers',
 'opel': 'Mean price 3911.00$ and 125495.00 mean Kilometers',
 'ford': 'Mean price 4939.00$ and 121253.00 mean Kilometers',
 'renault': 'Mean price 3297.00$ and 123307.00 mean Kilometers',
 'peugeot': 'Mean price 3772.00$ and 123362.00 mean Kilometers',
 'fiat': 'Mean price 3710.00$ and 110478.00 mean Kilometers',
 'skoda': 'Mean price 6739.00$ and 110430.00 mean Kilometers'}

| Brand | mean price in Dollars | mean kilometer amount |
| :-------------| :----- | :----- |
| audi | 10088 | 128012 |
| mercedes_benz | 9094 | 130461 |
| bmw | 8952 | 132200 |
| skoda | 6739 | 110430 |
| volkswagen | 6364 | 126561 |
| ford | 4939 | 121253 |
| opel | 3911 | 125495 |
| peugeot | 3772 | 123362 |
| fiat | 3710 | 110478 |
| renault | 3297 | 123307 |

Among the top 10 brands by listing volume, the premium German makers (audi, mercedes_benz, bmw) command the highest average prices even though their cars tend to have similar or higher mileage than the rest, which suggests brand reputation is driving price more than usage. Skoda and volkswagen sit in a middle tier, while opel, peugeot, fiat and renault form a cheaper group, with renault being the least expensive on average despite not having the highest mileage. This points to brand being a meaningful price driver on this dataset, independent of odometer reading.

## 8. Storing Aggregate Data in a DataFrame

The brand-level price and mileage figures from the previous section currently live in two separate plain dictionaries. Combining them into a single pandas DataFrame makes the numbers easier to read side by side and to reuse later, so that is what this section does.

In [93]:
brands_price_series = pd.Series(brands_price)
brands_price_series

,0
volkswagen,6364.0
bmw,8952.0
mercedes_benz,9094.0
audi,10088.0
opel,3911.0
ford,4939.0
renault,3297.0
peugeot,3772.0
fiat,3710.0
skoda,6739.0


In [94]:
brands_km_series = pd.Series(brands_km)
brands_km_series

,0
audi,128012.0
bmw,132200.0
fiat,110478.0
ford,121253.0
mercedes_benz,130461.0
opel,125495.0
peugeot,123362.0
renault,123307.0
skoda,110430.0
volkswagen,126561.0


In [95]:
frame = {'brand_mean_price':brands_price_series,
         'brand_mean_kilometers':brands_km_series}

output = pd.DataFrame(frame)
output

,brand_mean_price,brand_mean_kilometers
audi,10088.0,128012.0
bmw,8952.0,132200.0
fiat,3710.0,110478.0
ford,4939.0,121253.0
mercedes_benz,9094.0,130461.0
opel,3911.0,125495.0
peugeot,3772.0,123362.0
renault,3297.0,123307.0
skoda,6739.0,110430.0
volkswagen,6364.0,126561.0


This output table is now much easier to scan than the two separate dictionaries were: each row is a brand, and the two columns line up directly, making it simple to see at a glance, for example, that audi has both the highest average price and one of the higher mileages among the top 10 brands.

## 9. Next steps

In [96]:
car_cleaned['vehicle_type'].value_counts(dropna=False)

,count
vehicle_type,
limousine,11059
kombi,7891
kleinwagen,7613
bus,3797
cabrio,2893
coupe,2243
suv,1937
NaN,1482
andere,309


Several categorical columns (`vehicle_type`, `fuel_type`) use German short codes that are not obvious at a glance, so before moving on we translate them into more descriptive labels. Looking at `vehicle_type`, the largest categories are `limousine` (sedan-style body), `kombi` (estate/wagon) and `kleinwagen` (small city car); together with `bus`, `cabrio`, `coupe` and `suv` they cover almost all listings, with only ~1,500 rows missing a value and ~300 falling into the catch-all `andere` (other) bucket. We map each code to a clearer label below.

In [97]:
categorical_vehicle_type = car_cleaned['vehicle_type'].value_counts().index[:]
category_translator = {'bus':'monovolumen','limousine':'sedan','kleinwagen':'compacto','kombi':'familiar','coupe':'coupe','suv':'suv','cabrio':'cabrio','andere':'otros'}
for category in categorical_vehicle_type:
  bool_category = car_cleaned['vehicle_type'] == category
  car_cleaned.loc[bool_category,'vehicle_type'] = category_translator[category]
car_cleaned.head(5)

,date_crawled,name,seller,offer_type,price_in_dollars,abtest,vehicle_type,registration_year,gearbox,CV,model,odometer_km,registration_month,fuel_type,brand,unrepared_damage,ad_created,nr_pictures,postal_code,last_seen
0,2016-03-26 17:47:46,Peugeot_807_160_NAVTECH_ON_BOARD,privat,Angebot,5000,control,monovolumen,2004,manuell,158,andere,150000,3,lpg,peugeot,nein,2016-03-26 00:00:00,0,79588,2016-04-06 06:45:54
1,2016-04-04 13:38:56,BMW_740i_4_4_Liter_HAMANN_UMBAU_Mega_Optik,privat,Angebot,8500,control,sedan,1997,automatik,286,7er,150000,6,benzin,bmw,nein,2016-04-04 00:00:00,0,71034,2016-04-06 14:45:08
2,2016-03-26 18:57:24,Volkswagen_Golf_1.6_United,privat,Angebot,8990,test,sedan,2009,manuell,102,golf,70000,7,benzin,volkswagen,nein,2016-03-26 00:00:00,0,35394,2016-04-06 20:15:37
3,2016-03-12 16:58:10,Smart_smart_fortwo_coupe_softouch/F1/Klima/Pan...,privat,Angebot,4350,control,compacto,2007,automatik,71,fortwo,70000,6,benzin,smart,nein,2016-03-12 00:00:00,0,33729,2016-03-15 03:16:28
4,2016-04-01 14:38:50,Ford_Focus_1_6_Benzin_TÜV_neu_ist_sehr_gepfleg...,privat,Angebot,1350,test,familiar,2003,manuell,0,focus,150000,7,benzin,ford,nein,2016-04-01 00:00:00,0,39218,2016-04-01 14:38:50


### 9.1 fuel_type column

In [98]:
car_cleaned['fuel_type'].value_counts(dropna=False)

,count
fuel_type,
benzin,22913
diesel,13571
NaN,2026
lpg,586
cng,63
hybrid,37
elektro,17
andere,11


In [99]:
categorical_type_fuel = car_cleaned['fuel_type'].value_counts().index[:]
fuel_type_translator = {'benzin':'gasolina','diesel':'diesel','lpg':'lpg','cng':'cng','hybrid':'hibrido','elektro':'electrico','andere':'otros'}
for category in categorical_type_fuel:
  bool_fuel = car_cleaned['fuel_type'] == category
  car_cleaned.loc[bool_fuel,'fuel_type'] = fuel_type_translator[category]
car_cleaned['fuel_type'].value_counts(dropna=False)

,count
fuel_type,
gasolina,22913
diesel,13571
NaN,2026
lpg,586
cng,63
hibrido,37
electrico,17
otros,11


Petrol (`gasolina`) and diesel together power roughly 93% of the cleaned listings, while alternative fuels (`lpg`, `cng`, `hibrido`, `electrico`, `otros`) each make up less than 1.5% of the data individually. This confirms the dataset is overwhelmingly made up of conventional combustion-engine cars, so any brand- or price-level conclusions drawn later mostly describe petrol/diesel vehicles rather than the used-car market as a whole. The ~2,000 missing `fuel_type` values (about 5% of the cleaned data) are left as-is rather than guessed, since forcing them into a category could bias the fuel-type breakdown.

### 9.2 Convert date columns into numeric data

The three date-like text columns (`date_crawled`, `ad_created`, `last_seen`) are stored as full timestamps, but for aggregation purposes we only need the date part. We strip everything after the first 10 characters and remove the dashes so each becomes a plain `YYYYMMDD` integer-like string, which is easier to sort and group by later.

In [100]:
replace_columns = ['date_crawled', 'ad_created', 'last_seen']

for column in replace_columns:
  car_cleaned[column] = car_cleaned[column].str[:10]
  car_cleaned[column] = car_cleaned[column].str.replace('-','')

car_cleaned[['date_crawled','ad_created','last_seen']].head()

,date_crawled,ad_created,last_seen
0,20160326,20160326,20160406
1,20160404,20160404,20160406
2,20160326,20160326,20160406
3,20160312,20160312,20160315
4,20160401,20160401,20160401


### 9.3 Most common make/model combinations

For each brand we can look at which specific model shows up most often, which tells us what the "typical" car of that brand looks like in this market. We build a small helper that, given a brand, returns its top models by listing count.

In [101]:
def top_models_for_brand(brand):
  bool_brand = car_cleaned['brand'] == brand
  return car_cleaned.loc[bool_brand, 'model'].value_counts()[:5]
print(top_models_for_brand('audi'))
print(top_models_for_brand('mercedes_benz'))
print(top_models_for_brand('bmw'))

model
a4        1125
a3         788
a6         771
andere     212
tt         144
Name: count, dtype: int64
model
c_klasse    1064
e_klasse     911
a_klasse     476
andere       423
clk          236
Name: count, dtype: int64
model
3er        2353
5er        1083
1er         512
x_reihe     296
7er         124
Name: count, dtype: int64


Each premium German brand has one dominant "bread and butter" model line: the a4/a3/a6 family for audi, the c/e/a-class range for mercedes_benz, and the 3er/5er/1er series for bmw. In every case a single model line (audi's a4, mercedes' c_klasse, bmw's 3er) accounts for a noticeably larger share than the rest of that brand's catalog, suggesting these mid-range models are the ones sellers list most often on this marketplace.

### 9.4 Average price by mileage group

Grouping listings by their odometer reading lets us check whether price falls consistently as mileage increases, which is the pattern we would expect for used cars.

In [102]:
km_group = car_cleaned['odometer_km'].value_counts().index[:]
km_group

Index([150000, 125000, 100000,  90000,  80000,  70000,  60000,  50000,  40000,
        30000,  20000,   5000,  10000],
      dtype='int64', name='odometer_km')

In [103]:
avg_price_by_km = {}

for km in km_group:
  selected_km = car_cleaned[car_cleaned['odometer_km'] == km]
  mean_price = selected_km['price_in_dollars'].mean().round()
  avg_price_by_km[km] = mean_price
avg_price_by_km

{150000: np.float64(4601.0),
 125000: np.float64(6864.0),
 100000: np.float64(14213.0),
 90000: np.float64(9070.0),
 80000: np.float64(10115.0),
 70000: np.float64(11341.0),
 60000: np.float64(12893.0),
 50000: np.float64(15455.0),
 40000: np.float64(15808.0),
 30000: np.float64(17205.0),
 20000: np.float64(19753.0),
 5000: np.float64(21454.0),
 10000: np.float64(22487.0)}

Interestingly, average price does not decrease monotonically with mileage. Cars sitting at 150,000km (the odometer cap) are the cheapest on average (around 4,600 USD), while cars near 5,000-10,000km are the most expensive (around 21,000-22,500 USD), so the overall direction matches expectations. However, the relationship is not perfectly linear in the middle of the range: for example, 100,000km cars average a higher price (14,213 USD) than several lower-mileage brackets like 90,000km (9,070 USD) or 80,000km (10,115 USD). This is likely because mileage alone does not fully explain price. Vehicle age, brand mix and engine power also vary between these groups and are probably confounding the simple mileage-to-price relationship.

### 9.5 How much cheaper are damaged cars than their undamaged counterparts?

Using the same mileage buckets, we now repeat the average-price calculation but restricted to listings flagged `unrepared_damage == 'ja'` (damage not yet repaired), and compare it against the non-damaged averages computed above.

In [104]:
avg_price_by_km_damage = {}

for km in km_group:
  selected_km = car_cleaned[(car_cleaned['odometer_km'] == km) & (car_cleaned['unrepared_damage'] == 'ja')]
  mean_price = selected_km['price_in_dollars'].mean()
  avg_price_by_km_damage[km] = mean_price.round()
avg_price_by_km_damage


{150000: np.float64(2930.0),
 125000: np.float64(3841.0),
 100000: np.float64(4707.0),
 90000: np.float64(4405.0),
 80000: np.float64(6057.0),
 70000: np.float64(5648.0),
 60000: np.float64(7955.0),
 50000: np.float64(9357.0),
 40000: np.float64(13433.0),
 30000: np.float64(6899.0),
 20000: np.float64(7706.0),
 5000: np.float64(3289.0),
 10000: np.float64(4566.0)}

In [105]:
series_nondamage = pd.Series(avg_price_by_km)
df_nondamage = pd.DataFrame(series_nondamage).rename(columns={0:'price_not_damage'})

series_damage = pd.Series(avg_price_by_km_damage)
df_damage = pd.DataFrame(series_damage).rename(columns={0:'price_damage'})

series_diference = series_nondamage - series_damage
df_diference = pd.DataFrame(series_diference).rename(columns={0:'price_difference'})

series_diference_percent = (series_damage * 100) / series_nondamage.round(2)
df_diference_percent = pd.DataFrame(series_diference_percent).rename(columns={0:'percentage_difference'})

df = pd.concat([df_nondamage, df_damage, df_diference, df_diference_percent], axis=1)
df

,price_not_damage,price_damage,price_difference,percentage_difference
150000,4601.0,2930.0,1671.0,63.681808
125000,6864.0,3841.0,3023.0,55.958625
100000,14213.0,4707.0,9506.0,33.117568
90000,9070.0,4405.0,4665.0,48.566703
80000,10115.0,6057.0,4058.0,59.881364
70000,11341.0,5648.0,5693.0,49.801605
60000,12893.0,7955.0,4938.0,61.700147
50000,15455.0,9357.0,6098.0,60.543513
40000,15808.0,13433.0,2375.0,84.975962
30000,17205.0,6899.0,10306.0,40.098808


| odometer_km | price_not_damage | price_damage | price_difference | percentage_difference |
| :--- | :--- | :--- | :--- | :--- |
| 150000 | 4601 | 2930 | 1671 | 63.68% |
| 125000 | 6864 | 3841 | 3023 | 55.96% |
| 100000 | 14213 | 4707 | 9506 | 33.12% |
| 90000 | 9070 | 4405 | 4665 | 48.57% |
| 80000 | 10115 | 6057 | 4058 | 59.88% |
| 70000 | 11341 | 5648 | 5693 | 49.80% |
| 60000 | 12893 | 7955 | 4938 | 61.70% |
| 50000 | 15455 | 9357 | 6098 | 60.54% |
| 40000 | 15808 | 13433 | 2375 | 84.98% |
| 30000 | 17205 | 6899 | 10306 | 40.10% |
| 20000 | 19753 | 7706 | 12047 | 39.01% |
| 5000 | 21454 | 3289 | 18165 | 15.33% |
| 10000 | 22487 | 4566 | 17921 | 20.31% |

Across every mileage bucket, damaged cars sell for noticeably less than comparable undamaged ones, but the size of the discount varies a lot. On average, damaged listings go for roughly 15% to 65% of what an equivalent undamaged car fetches, meaning the "damage discount" ranges from about 35% up to nearly 85% of the original value depending on mileage. Low-mileage cars (5,000-10,000km) suffer the steepest relative discount when damaged (they retain only 15-20% of their undamaged value), which makes sense: buyers expect a low-mileage car to be pristine, so visible unrepaired damage is a bigger red flag on an otherwise "like new" vehicle. Higher-mileage cars (40,000-150,000km) tend to keep a larger share of their value even when damaged (50-85%), since buyers already expect wear at that mileage and the marginal impact of damage on perceived quality is smaller. This confirms that both mileage and damage status are meaningful, independent price drivers, and that their effects interact rather than simply adding up.

## 10. Overall conclusion

Cleaning this dataset removed a small but important slice of bad data: a handful of obviously fake prices at the top (Ferraris and Volkswagens "worth" tens of millions of dollars), listings priced below 850 that added noise without much analytical value, and registration years outside the plausible 1927-2016 window. After these steps the working set shrank from 50,000 to about 39,200 usable listings, which is still large enough to draw meaningful conclusions.

On pricing, three separate factors stood out as real, independent drivers rather than noise: brand (premium German marques command a consistent premium over mass-market brands), mileage (price generally falls as mileage rises, though not perfectly linearly), and repair status (unrepaired damage cuts the average price everywhere, and does so proportionally more on low-mileage cars than on high-mileage ones). Fuel type and vehicle type were useful for describing the market (mostly petrol and diesel, mostly sedans/wagons/small cars) but did not need to be cleaned further for the price analysis.

If this analysis were extended further, the most useful next step would be a multi-variable model (for example a simple linear regression on brand, mileage, registration year, power and damage status together) so we can quantify how much of the price variation each factor explains once the others are controlled for, rather than looking at each one individually as we did here.